# SPDAP Demo Notebook

Notebook Overview
This Jupyter Notebook helps users easily select XML files that contain wafer measurement data and run only the analysis they need.

It provides an interactive UI for selecting XML files from the data/ folder.

Users can filter files by wafer, timestamp, die coordinate, keyword, and file name.

Analysis results are shown as previews first, so unnecessary output files are not created.

CSV files, analysis figures, and wafermaps are saved only when the user clicks Save.


- Select one XML file from the filters.
- Press **Parse Selected XML** to load and parse the data.
- Review the key parameter table.
- The CSV preview appears automatically. Press **Generate Figure** or **Generate Wafermap** to inspect those outputs (nothing is written to disk).
- Press **Save** to write CSV + previewed figures to `res/jupyter_notebook/<wafer>/<timestamp>/`.


In [4]:
from pathlib import Path
import re
import shutil
import sys
import tempfile

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "run.py").exists()),
    Path.cwd(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError(
        "ipywidgets is required for this notebook. Install it with `pip install ipywidgets`."
    ) from exc

import importlib

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, clear_output, Image

# Import all src.* modules first, then reload bottom-up so edits in src/ are picked up
# without restarting the kernel. Order matters: dependencies before aggregators.
import src.config as config_module
import src.csv_export as csv_export_module
import src.wafermap as wafermap_module
import src.iv_analysis as iv_analysis_module
import src.main as main_module

config_module = importlib.reload(config_module)
csv_export_module = importlib.reload(csv_export_module)
wafermap_module = importlib.reload(wafermap_module)
iv_analysis_module = importlib.reload(iv_analysis_module)
main_module = importlib.reload(main_module)

# Re-fetch names from the freshly reloaded modules
from src.csv_export import rows_to_dataframe, summarize_xml, write_csv
from src.wafermap import plot_wafermap
from src.config import DATA_DIR
analyze_figure = main_module.analyze_figure

WHITE_RC = {
    "figure.figsize": (10, 5),
    "figure.facecolor": "white",
    "figure.edgecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "black",
    "axes.labelcolor": "black",
    "axes.titlecolor": "black",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
    "text.color": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "grid.color": "gray",
}

def force_white_style():
    """Re-apply white style; call before any src.* plotting function."""
    plt.style.use("default")
    plt.rcParams.update(WHITE_RC)

def whiten_png(png_path):
    """Composite a saved PNG onto a solid white background (handles transparent saves)."""
    try:
        from PIL import Image as _PIL
    except ImportError:
        return
    img = _PIL.open(png_path)
    if img.mode in ("RGBA", "LA"):
        bg = _PIL.new("RGB", img.size, (255, 255, 255))
        mask = img.split()[-1]
        bg.paste(img.convert("RGB"), mask=mask)
        bg.save(png_path)
    img.close()

force_white_style()

DATA_ROOT = PROJECT_ROOT / DATA_DIR
# Inline previews go to a per-session tempdir. Nothing is written under res/ until Save is pressed.
PREVIEW_DIR = Path(tempfile.mkdtemp(prefix="spdap_preview_"))



In [5]:
xml_files = sorted(path for path in DATA_ROOT.rglob("*.xml") if "LMZ" in path.name.upper())

file_pattern = re.compile(r"\((-?\d+),(-?\d+)\)")
records = []
for xml_path in xml_files:
    wafer = xml_path.parent.parent.name if xml_path.parent.parent != DATA_ROOT else ""
    timestamp = xml_path.parent.name
    die_column, die_row = "", ""
    match = file_pattern.search(xml_path.stem)
    if match:
        die_column, die_row = match.group(1), match.group(2)
    die = f"({die_column},{die_row})" if die_column and die_row else ""
    records.append(
        {
            "xml_path": str(xml_path),
            "wafer": wafer,
            "timestamp": timestamp,
            "die_column": die_column,
            "die_row": die_row,
            "die": die,
            "file_name": xml_path.name,
            "device_type": "LMZ",
        }
    )

files_df = pd.DataFrame(records)

print(f"Total LMZ XML files found: {len(files_df)}")
if len(files_df) == 0:
    display(widgets.HTML("<b>데이터 디렉터리에 XML 파일이 없습니다. data/ 폴더를 확인하세요.</b>"))
else:
    display(files_df.head(10))

Total LMZ XML files found: 98


,xml_path,wafer,timestamp,die_column,die_row,die,file_name,device_type
0,C:\Users\혁\PycharmProjects\project\data\HY2021...,D07,20190715_190855,-1,-1,"(-1,-1)","HY202103_D07_(-1,-1)_LION1_DCM_LMZC.xml",LMZ
1,C:\Users\혁\PycharmProjects\project\data\HY2021...,D07,20190715_190855,-1,-3,"(-1,-3)","HY202103_D07_(-1,-3)_LION1_DCM_LMZC.xml",LMZ
2,C:\Users\혁\PycharmProjects\project\data\HY2021...,D07,20190715_190855,-1,3,"(-1,3)","HY202103_D07_(-1,3)_LION1_DCM_LMZC.xml",LMZ
3,C:\Users\혁\PycharmProjects\project\data\HY2021...,D07,20190715_190855,-3,-3,"(-3,-3)","HY202103_D07_(-3,-3)_LION1_DCM_LMZC.xml",LMZ
4,C:\Users\혁\PycharmProjects\project\data\HY2021...,D07,20190715_190855,-3,0,"(-3,0)","HY202103_D07_(-3,0)_LION1_DCM_LMZC.xml",LMZ
5,C:\Users\혁\PycharmProjects\project\data\HY2021...,D07,20190715_190855,-3,2,"(-3,2)","HY202103_D07_(-3,2)_LION1_DCM_LMZC.xml",LMZ
6,C:\Users\혁\PycharmProjects\project\data\HY2021...,D07,20190715_190855,-4,-1,"(-4,-1)","HY202103_D07_(-4,-1)_LION1_DCM_LMZC.xml",LMZ
7,C:\Users\혁\PycharmProjects\project\data\HY2021...,D07,20190715_190855,0,-4,"(0,-4)","HY202103_D07_(0,-4)_LION1_DCM_LMZC.xml",LMZ
8,C:\Users\혁\PycharmProjects\project\data\HY2021...,D07,20190715_190855,0,0,"(0,0)","HY202103_D07_(0,0)_LION1_DCM_LMZC.xml",LMZ
9,C:\Users\혁\PycharmProjects\project\data\HY2021...,D07,20190715_190855,0,2,"(0,2)","HY202103_D07_(0,2)_LION1_DCM_LMZC.xml",LMZ


## 1) Select Data and Run Analysis

Choose wafer, timestamp, die column, die row, keyword, and XML file. Parsing does not run automatically. Press **Parse Selected XML** to show the parsed table. After parsing, the preview buttons and **Save** become available.


In [ ]:

current_rows = []
current_xml_path = None

wafer_dropdown = widgets.Dropdown(
    options=[(value, value) for value in sorted(files_df["wafer"].dropna().unique())],
    description="Wafer",
    layout=widgets.Layout(width="260px"),
)
timestamp_dropdown = widgets.Dropdown(
    options=[],
    description="Timestamp",
    layout=widgets.Layout(width="260px"),
)
die_dropdown = widgets.Dropdown(
    options=[],
    description="Die",
    layout=widgets.Layout(width="260px"),
)
keyword_text = widgets.Text(
    value="",
    description="Keyword",
    placeholder="file name, LMZC, D08, (0,0), etc.",
    layout=widgets.Layout(width="520px"),
)
file_dropdown = widgets.Dropdown(
    options=[],
    description="XML file",
    layout=widgets.Layout(width="760px"),
)

parse_button = widgets.Button(description="Parse Selected XML", button_style="primary", icon="search")
figure_button = widgets.Button(description="Figure", button_style="info", icon="bar-chart")
wafermap_button = widgets.Button(description="Wafermap", button_style="warning", icon="th")
save_button = widgets.Button(description="Save", button_style="success", icon="save")
figure_button.disabled = True
wafermap_button.disabled = True
save_button.disabled = True
message_area = widgets.HTML(value="<i>Waiting for XML selection.</i>")
figure_output = widgets.Output(layout={"border": "1px solid #444", "padding": "8px"})
wafermap_output = widgets.Output(layout={"border": "1px solid #444", "padding": "8px"})
csv_preview_output = widgets.Output(layout={"border": "1px solid #444", "padding": "8px"})

_updating_options = False


def keyword_filtered(df):
    keyword = keyword_text.value.strip().lower()
    if not keyword:
        return df
    return df[df.apply(
        lambda row: keyword in str(row["file_name"]).lower()
        or keyword in str(row["wafer"]).lower()
        or keyword in str(row["timestamp"]).lower()
        or keyword in str(row["die"]).lower()
        or keyword in str(row["die_column"]).lower()
        or keyword in str(row["die_row"]).lower(),
        axis=1,
    )]


def base_filtered_files(include_die=True):
    filtered = keyword_filtered(files_df.copy())
    if wafer_dropdown.value:
        filtered = filtered[filtered["wafer"] == wafer_dropdown.value]
    if timestamp_dropdown.value:
        filtered = filtered[filtered["timestamp"] == timestamp_dropdown.value]
    if include_die and die_dropdown.value:
        filtered = filtered[filtered["die"] == die_dropdown.value]
    return filtered


def set_dropdown_options(dropdown, options):
    current = dropdown.value
    dropdown.options = options
    option_values = [option[1] if isinstance(option, tuple) else option for option in options]
    if current in option_values:
        dropdown.value = current
    elif option_values:
        dropdown.value = option_values[0]
    else:
        dropdown.value = None


def update_timestamp_options():
    filtered = keyword_filtered(files_df.copy())
    if wafer_dropdown.value:
        filtered = filtered[filtered["wafer"] == wafer_dropdown.value]
    timestamps = [(value, value) for value in sorted(v for v in filtered["timestamp"].dropna().unique() if v)]
    set_dropdown_options(timestamp_dropdown, timestamps)


def _die_sort_key(value):
    match = re.fullmatch(r"\((-?\d+),(-?\d+)\)", str(value))
    return (int(match.group(1)), int(match.group(2))) if match else (999999, str(value))


def update_die_options():
    filtered = base_filtered_files(include_die=False)
    dies = [(value, value) for value in sorted((v for v in filtered["die"].dropna().unique() if v), key=_die_sort_key)]
    set_dropdown_options(die_dropdown, dies)


def update_file_options():
    filtered = base_filtered_files(include_die=True)
    options = [(Path(path).name, str(path)) for path in filtered["xml_path"].tolist()]
    file_dropdown.options = options
    file_dropdown.value = options[0][1] if options else None
    reset_analysis_state()
    render_current_selection()


def reset_analysis_state():
    global current_rows, current_xml_path
    current_rows = []
    current_xml_path = None
    figure_button.disabled = True
    wafermap_button.disabled = True
    save_button.disabled = True
    global preview_figure_path, preview_wafermap_path, preview_csv_path
    preview_figure_path = None
    preview_wafermap_path = None
    preview_csv_path = None
    figure_output.clear_output()
    wafermap_output.clear_output()
    csv_preview_output.clear_output()


def refresh_dependent_options(change=None):
    global _updating_options
    if _updating_options:
        return
    _updating_options = True
    try:
        update_timestamp_options()
        update_die_options()
        update_file_options()
    finally:
        _updating_options = False




def refresh_files_only(change=None):
    if not _updating_options:
        update_file_options()


def render_current_selection():
    selected = file_dropdown.value
    if not selected:
        message_area.value = "<b>No XML file matches the current filters.</b>"
        parse_button.disabled = True
        return
    path = Path(selected)
    message_area.value = f"<b>Selected XML:</b> {path.name}<br><i>Press Parse Selected XML to run parsing.</i>"
    parse_button.disabled = False




def default_output_dir(xml_path):
    return PROJECT_ROOT / "res" / "jupyter_notebook" / xml_path.parent.parent.name / xml_path.parent.name


def parse_selected_xml(_button=None):
    global current_rows, current_xml_path
    global preview_figure_path, preview_wafermap_path

    selected = file_dropdown.value
    if not selected:
        message_area.value = "<b>Select an XML file before parsing.</b>"
        return

    parse_button.disabled = True
    figure_button.disabled = True
    wafermap_button.disabled = True
    save_button.disabled = True
    figure_output.clear_output()
    wafermap_output.clear_output()
    csv_preview_output.clear_output()
    preview_figure_path = None
    preview_wafermap_path = None

    current_xml_path = Path(selected)
    try:
        current_rows = summarize_xml(current_xml_path)
    except Exception as exc:
        message_area.value = f"<b>XML parsing failed:</b> {exc}"
        parse_button.disabled = False
        return

    if current_rows:
        display_csv_preview()
        message_area.value = f"<b>Parsing completed:</b> {current_xml_path.name}"
    else:
        message_area.value = (
            f"<b>Parsing completed:</b> {current_xml_path.name}"
            f"<br><i>No MZM summary rows were parsed from this XML file.</i>"
        )

    parse_button.disabled = False
    figure_button.disabled = False
    wafermap_button.disabled = False
    save_button.disabled = False

def generate_current_figure(_button=None):
    global preview_figure_path
    if current_xml_path is None:
        message_area.value = "<b>Parse XML before generating the figure.</b>"
        return

    figure_button.disabled = True
    with figure_output:
        clear_output(wait=True)
        png_path = PREVIEW_DIR / f"{current_xml_path.stem}_analysis.png"
        png_path.parent.mkdir(parents=True, exist_ok=True)
        try:
            force_white_style()
            success = analyze_figure(current_xml_path, png_path)
            if success and png_path.exists():
                whiten_png(png_path)
        except Exception as exc:
            print(f"Figure preview failed: {exc}")
            preview_figure_path = None
            figure_button.disabled = False
            return

        if success and png_path.exists():
            preview_figure_path = png_path
            print("Figure preview (not saved yet). Press Save to write to disk.")
            display(Image(filename=str(png_path)))
        else:
            preview_figure_path = None
            print("No analysis figure was generated for the selected XML file.")

    figure_button.disabled = False


def generate_current_wafermap(_button=None):
    global preview_wafermap_path
    if current_xml_path is None:
        message_area.value = "<b>Parse XML before generating the wafermap.</b>"
        return

    wafer = current_xml_path.parent.parent.name
    timestamp = current_xml_path.parent.name
    xml_files = sorted(current_xml_path.parent.glob("*LMZ*.xml"))
    wafermap_button.disabled = True

    with wafermap_output:
        clear_output(wait=True)
        print(f"Generating wafermap preview for {wafer} / {timestamp}")
        print(f"XML files: {len(xml_files)}")
        rows = []
        for xml_path in xml_files:
            try:
                rows.extend(summarize_xml(xml_path))
            except Exception as exc:
                print(f"Skipped {xml_path.name}: {exc}")

        if not rows:
            print("No rows were available for wafermap generation.")
            preview_wafermap_path = None
            wafermap_button.disabled = False
            return

        png_path = PREVIEW_DIR / f"{wafer}_{timestamp}_wafermap.png"
        try:
            force_white_style()
            success = plot_wafermap(wafer, timestamp, rows, png_path)
            if success and png_path.exists():
                whiten_png(png_path)
        except Exception as exc:
            print(f"Wafermap preview failed: {exc}")
            preview_wafermap_path = None
            wafermap_button.disabled = False
            return

        if success and png_path.exists():
            preview_wafermap_path = png_path
            print("Wafermap preview (not saved yet). Press Save to write to disk.")
            display(Image(filename=str(png_path)))
        else:
            preview_wafermap_path = None
            print("No wafermap was generated. Check that die coordinates and wafermap parameters exist.")

    wafermap_button.disabled = False


def display_csv_preview():
    global preview_csv_path
    csv_temp = PREVIEW_DIR / f"{current_xml_path.stem}.csv"
    try:
        write_csv(csv_temp, current_rows)
        df_preview = rows_to_dataframe(current_rows)
    except Exception as exc:
        preview_csv_path = None
        with csv_preview_output:
            clear_output(wait=True)
            print(f"CSV preview failed: {exc}")
        return

    preview_csv_path = csv_temp
    with csv_preview_output:
        clear_output(wait=True)
        display(widgets.HTML(
            f"<b>CSV preview (not saved yet).</b> {len(df_preview)} rows x {len(df_preview.columns)} columns. Press Save to write to disk."
        ))
        display(df_preview)

def save_current_outputs(_button=None):
    if current_xml_path is None:
        message_area.value = "<b>Parse XML before saving.</b>"
        return
    if not current_rows:
        message_area.value = "<b>No parsed rows to save.</b>"
        return

    save_button.disabled = True
    out_dir = default_output_dir(current_xml_path)
    out_dir.mkdir(parents=True, exist_ok=True)

    saved = []
    errors = []

    csv_out = out_dir / f"{current_xml_path.stem}.csv"
    if preview_csv_path and preview_csv_path.exists():
        try:
            shutil.copy2(preview_csv_path, csv_out)
            saved.append(f"CSV: {csv_out.relative_to(PROJECT_ROOT)}")
        except Exception as exc:
            errors.append(f"CSV: {exc}")
    else:
        errors.append("CSV: no preview to copy (re-parse the XML)")

    if preview_figure_path and preview_figure_path.exists():
        png_out = out_dir / f"{current_xml_path.stem}_analysis.png"
        try:
            shutil.copy2(preview_figure_path, png_out)
            saved.append(f"Figure: {png_out.relative_to(PROJECT_ROOT)}")
        except Exception as exc:
            errors.append(f"Figure: {exc}")

    if preview_wafermap_path and preview_wafermap_path.exists():
        wm_out = out_dir / "wafermap.png"
        try:
            shutil.copy2(preview_wafermap_path, wm_out)
            saved.append(f"Wafermap: {wm_out.relative_to(PROJECT_ROOT)}")
        except Exception as exc:
            errors.append(f"Wafermap: {exc}")

    if saved and not errors:
        message_area.value = "<b>Save completed</b><br>" + "<br>".join(saved)
    elif saved and errors:
        message_area.value = (
            "<b>Save partially completed</b><br>"
            + "<br>".join(saved)
            + "<br><b>Errors:</b> " + "; ".join(errors)
        )
    else:
        message_area.value = "<b>Save failed:</b> " + "; ".join(errors)

    save_button.disabled = False


wafer_dropdown.observe(refresh_dependent_options, names="value")
timestamp_dropdown.observe(refresh_dependent_options, names="value")
keyword_text.observe(refresh_dependent_options, names="value")
die_dropdown.observe(refresh_files_only, names="value")
parse_button.on_click(parse_selected_xml)
figure_button.on_click(generate_current_figure)
wafermap_button.on_click(generate_current_wafermap)
save_button.on_click(save_current_outputs)

refresh_dependent_options()

filters_box = widgets.VBox([
    widgets.HBox([wafer_dropdown, timestamp_dropdown]),
    die_dropdown,
    keyword_text,
    file_dropdown,
    message_area,
    widgets.HBox([parse_button, figure_button, wafermap_button, save_button]),
    widgets.HTML("<i>Generate previews, then press Save to write to disk.</i>"),
    widgets.HTML("<b>CSV preview</b>"),
    csv_preview_output,
    widgets.HTML("<b>Figure preview</b>"),
    figure_output,
    widgets.HTML("<b>Wafermap preview</b>"),
    wafermap_output,
])

display(filters_box)


## 2) Execution Order

1. Run the setup/indexing cells.
2. Select wafer/timestamp/die/file.
3. Press **Parse Selected XML** to parse and display the table.
4. CSV preview appears automatically after parse. Use **Generate Figure** / **Generate Wafermap** for the other previews.
5. Press **Save** to write CSV + previewed figures to `res/jupyter_notebook/...`.
